# Energy Algorithms - Optimization Portfolio Walkthrough

**Target Audience:** Junior Optimization Engineer / algorithmic trading

**Modules:** Energy Markets (PCR/Euphemia), LP/MIP Optimization, Backtesting, ENTSO-E Data

## Setup - Imports

In [ ]:
import numpy as np

from energy_algorithms.adapters.entsoe_client import (
    fetch_demo_day_ahead,
    fetch_demo_generation_mix,
)
from energy_algorithms.domain.markets.intraday import demo_intraday
from energy_algorithms.domain.markets.market_clearing import demo_clearing
from energy_algorithms.domain.markets.multi_zone import demo_multi_zone
from energy_algorithms.domain.markets.pcr_model import PCRModel
from energy_algorithms.domain.optimization.portfolio import demo_portfolio
from energy_algorithms.domain.optimization.scheduling import demo_uc
from energy_algorithms.domain.optimization.storage import demo_storage
from energy_algorithms.domain.optimization.transportation import demo_transportation
from energy_algorithms.domain.trading.backtest_engine import backtest
from energy_algorithms.domain.trading.mean_reversion import mean_reversion
from energy_algorithms.domain.trading.momentum import momentum
from energy_algorithms.domain.trading.sma_crossover import sma_crossover

print("All modules imported")


## 1. Energy Markets - PCR & Euphemia Connection

### 1.1 Market Clearing

In [ ]:
r = demo_clearing()
print(
    f"Price: EUR {r['clearing_price']:.0f}/MWh | "
    f"Volume: {r['clearing_volume']:.0f} MWh"
)
print("Supply and demand stack equilibrium computed")


### 1.2 PCR Model - Block Orders

In [ ]:
m = PCRModel(area="IT")
m.add_supply("wind", 5, 300)
m.add_supply("gas", 80, 200)
m.add_supply("diesel", 120, 100)
m.add_demand("base", 200, 350)
m.add_demand("peak", 150, 150)
m.add_block("hydro_A", 35, 100, group="hydro")
m.add_block("hydro_B", 40, 50, group="hydro")
m.add_block("cfg1", 60, 80, group="excl_A")
m.add_block("cfg2", 55, 80, group="excl_A")
r = m.solve()
m.report()
print(f"MCP: EUR {r['mcp']:.0f}/MWh | Welfare: EUR {r['welfare']:,.0f}")


### 1.3 Multi-Zone Coupling - ATC Flows

In [ ]:
r = demo_multi_zone()
print(f"Welfare: EUR {r['welfare']:,.0f}")
for flow_name, mw in r["flows"].items():
    print(f"  {flow_name}: {mw} MW")
for zone_name, zone_data in r["zones"].items():
    print(f"  {zone_name}: MCP=EUR {zone_data['mcp']}/MWh")


### 1.4 Intraday Continuous Trading

In [ ]:
r = demo_intraday()
print(
    f"{len(r['trades'])} trades | {r['total_volume']} MW | "
    f"VWAP: EUR {r['vwap']:.2f}"
)
for trade in r["trades"][:5]:
    print(
        f"  t={trade['time']:5.1f}h EUR {trade['price']:7.1f} "
        f"x {trade['qty']:5.0f} MW"
    )


## 2. LP/MIP Optimization

### 2.1 Transportation

In [ ]:
r = demo_transportation()
print(f"Cost: EUR {r['total_cost']:,.0f}")
for (warehouse, route), quantity in r["allocations"].items():
    print(f"  {warehouse} -> {route}: {quantity:.0f}")


### 2.2 Portfolio Optimization - Mean-Variance

In [ ]:
r = demo_portfolio()
print(
    f"Return: {r['return']:.2%} | Risk: {r['risk']:.2%} | "
    f"Sharpe: {r['return'] / r['risk']:.2f}"
)
for i, weight in enumerate(r["weights"]):
    if weight > 0.001:
        print(f"  Asset {i + 1}: {weight:>6.1%}")


### 2.3 Unit Commitment - MIP

In [ ]:
r = demo_uc()
print(f"Cost: EUR {r['total_cost']:,.0f}")
for time_key, period in r["schedule"].items():
    t = int(time_key.split("=")[1])
    online = period["_online"]
    generation = sum(value for key, value in period.items() if not key.startswith("_"))
    print(
        f"  t={t:>2}: demand={period['_demand']:5.0f} "
        f"gen={generation:5.0f} online={online}"
    )


### 2.4 BESS Storage Optimization

In [ ]:
r = demo_storage()
print(f"Revenue: EUR {r['revenue']:,.2f} | Cycles: {r.get('total_cycles', 'N/A')}")
for hour, period in enumerate(r["schedule"][:12]):
    print(
        f"  H{hour:>2}: chg={period['charge']:6.1f} "
        f"dis={period['discharge']:6.1f} SoC={period['soc']:6.1f}"
    )
print("  ... (12 more hours)")


## 3. Backtesting Engine - Vectorized, No Look-Ahead Bias

In [ ]:
np.random.seed(42)
n = 252 * 2
prices = 100 * np.cumprod(1 + np.random.normal(0.0002, 0.015, n))
strategies = [
    ("Momentum", momentum(prices)),
    ("MeanRev", mean_reversion(prices)),
    ("SMA", sma_crossover(prices)),
]
for name, signal in strategies:
    r = backtest(prices, signal)
    print(
        f"  {name:<12} Ret={r['total_return']:>7.1%} "
        f"Sharpe={r['sharpe']:>6.2f} "
        f"MaxDD={r['max_drawdown']:>7.1%} Trades={r['n_trades']}"
    )


## 4. ENTSO-E Data Pipeline

In [ ]:
p = fetch_demo_day_ahead()
print(
    f"{p['area']} {p['date']}: Avg EUR {p['avg_price']}/MWh | "
    f"Range EUR {p['min_price']}-{p['max_price']}"
)
g = fetch_demo_generation_mix()
print(f"Generation mix ({g['total_mw']} MW):")
for source in g["generation"]:
    share = source["mw"] / g["total_mw"] * 100
    print(f"  {source['type']:<25} {source['mw']:>6.0f}MW ({share:>5.1f}%)")


## Summary

| Skill | Module |
|-------|--------|
| Social welfare LP | `energy_algorithms/domain/markets/pcr_model.py` |
| Block orders | `energy_algorithms/domain/markets/block_orders.py` |
| Multi-zone coupling | `energy_algorithms/domain/markets/multi_zone.py` |
| Intraday simulation | `energy_algorithms/domain/markets/intraday.py` |
| Unit commitment MIP | `lp_optimization/scheduling.py` |
| BESS storage LP | `lp_optimization/storage.py` |
| Portfolio optimization | `lp_optimization/portfolio.py` |
| Vectorized backtesting | `backtester/engine.py` |
| ENTSO-E pipeline | `energy_data/fetcher.py` |
| 40 tests | `tests/` |

**Run all tests:** `pytest tests/ -v` (40 tests passing)

**For interviewers:** See `energy_algorithms/domain/markets/interview_prep.md`